# 02 · Data Acquisition — PlantVillage

Downloads the PlantVillage colour images for the 10 classes in scope, plus the
`leaf_grouping` maps (needed later so images of the same physical leaf never
land in both train and test).

The repository is cloned to **Colab's local disk** and only the finished class
folders are copied to Drive. Running Git directly on the Drive mount caused the
earlier failures: stale `index.lock` files and a half-checked-out
`Tomato___Late_blight` folder with 0 of its 1,909 images.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/crop-disease-detection-and-marketplace.git"
REPO_DIR = Path("/content/drive/MyDrive/crop-disease-detection")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

!git -C {REPO_DIR} status --short --branch

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

## Remove the earlier broken clone

The old folder on Drive is a Git working copy stuck mid-checkout. PlantVillage
is public and re-downloadable, so it is safe to delete. Set the flag to `True`
to remove it.

In [ ]:
import shutil

REMOVE_OLD_CLONE = False

old_clone = config.PLANTVILLAGE_DIR
if (old_clone / ".git").exists():
    print("Old clone found:", old_clone)
    if REMOVE_OLD_CLONE:
        shutil.rmtree(old_clone)
        print("Removed.")
    else:
        print("Set REMOVE_OLD_CLONE = True and re-run to remove it.")
else:
    print("No old clone on Drive.")

## Clone the target classes to local disk

In [ ]:
from src.data import plantvillage

LOCAL_CLONE = Path("/content/plantvillage_src")

assert not (config.PLANTVILLAGE_DIR / ".git").exists(), (
    "The old broken clone is still on Drive - remove it in the cell above first."
)

plantvillage.clone_sparse(LOCAL_CLONE, config.PLANTVILLAGE_CLASSES)
print("Cloned to", LOCAL_CLONE)

## Copy to Drive and verify against Git

In [ ]:
import pandas as pd

report = pd.DataFrame(
    plantvillage.copy_to_destination(
        LOCAL_CLONE, config.PLANTVILLAGE_CLASSES, config.PLANTVILLAGE_DIR
    )
)
report["crop"] = report["class"].str.split("___").str[0].replace({"Corn_(maize)": "Maize"})

display(report[["crop", "class", "expected", "copied", "ok"]])
print()
print(report.groupby("crop")["copied"].sum().to_string())
print("\nTotal:", report["copied"].sum())

assert report["ok"].all(), "Some classes are incomplete - see rows where ok is False."
print("All classes complete.")

## Save the inventory

A CSV snapshot of what was acquired, for the data preparation report.

In [ ]:
inventory_path = config.METADATA_DIR / "plantvillage_inventory.csv"
report.to_csv(inventory_path, index=False)
print("Saved", inventory_path)

## Save to GitHub

Commits everything changed in the repository on Drive (code and notebooks only —
datasets are outside the repository) and pushes the current branch.

Needs two Colab secrets (key icon, left sidebar): `GITHUB_TOKEN` (a GitHub
personal access token with repo access) and `GIT_EMAIL`. Save the notebook
(Ctrl+S) before running this cell.

In [ ]:
from google.colab import userdata

from src import github_sync

print(github_sync.status(REPO_DIR))
print()
print(github_sync.push(
    REPO_DIR,
    message="Acquire PlantVillage target classes",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
))